# <h1 style='background:#3b3a30; border:2; border-radius: 10px;padding-top: 2%;; font-size:200%; font-weight: bold; color:#c1502e'><center>Introduction</center></h1> 

This Provided Dataset was generated from a deep learning model trained on the[ California housing dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html) .
> **Contents Of This Dataset:**
-  `MedInc` - median income in block group
-  `HouseAge` - median house age in block group
-  `AveRooms` - average number of rooms per household
-  `AveBedrms` - average number of bedrooms per household
-  `Population` - block group population
-  `AveOccup` - average number of household members
-  `Latitude` - block group latitude
-  `Longitude` - block group longitude

**Our Task** is to to predict `MedHouseVal` .

# <h1 style='background:#3b3a30; border:2; border-radius: 10px;padding-top: 2%;; font-size:200%; font-weight: bold; color:#c1502e'><center>Importing Libraies</center></h1> 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.cluster import DBSCAN
from sklearn.neighbors import LocalOutlierFactor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgbm
from lightgbm.sklearn import LGBMRegressor


In [ ]:
df_train=pd.read_csv("../input/playground-series-s3e1/train.csv")
df_test=pd.read_csv("../input/playground-series-s3e1/test.csv")
df_sub=pd.read_csv("../input/playground-series-s3e1/sample_submission.csv")

In [ ]:
print(f"shape of training set:{df_train.shape}")
print(f"shape of training set:{df_test.shape}")
print(f"shape of training set:{df_sub.shape}")

In [ ]:
df_train.head()

# <h1 style='background:#3b3a30; border:2; border-radius: 10px;padding-top: 2%;; font-size:200%; font-weight: bold; color:#c1502e'><center>Checking For Missing Values</center></h1> 

In [ ]:
df_train.isna().sum()

**So We have No Null Value Here .**

# <h1 style='background:#3b3a30; border:2; border-radius: 10px;padding-top: 2%;; font-size:200%; font-weight: bold; color:#c1502e'><center>Exploring Dataset</center></h1> 

In [ ]:
df_train.info()

In [ ]:
# Checking values of cols

for i in df_train.columns:
    print("column name {} and present unique values are {}".format(i,len(df_train[i].unique())))

So it seems like we no category columns . All are Numeric .

In [ ]:
# Finding Co-relation

num_feat=df_train.select_dtypes(include=[np.number])
corr=num_feat.corr()
f,ax=plt.subplots(figsize=(14,7))
plt.title("correlation",y=1,size=16)
sns.heatmap(corr,square=True,vmax=0.7)

We have to keep in mind that **MedHouseVal** is our target col . So from this heatmap we are getting to know that **'MedInc','HouseAge', 'AveRooms'** columns are more correlated with our target col .

In [ ]:
#loooking for some relations

sns.scatterplot(x='MedInc',y='MedHouseVal',data=df_train)

In [ ]:
# Some Outliers are present

sns.scatterplot(x='AveRooms',y='MedHouseVal',data=df_train)

In [ ]:
# Looking for Distribution of cols

n_bins = 50
histplot_hyperparams = {
    'kde':True,
    'alpha':0.4,
    'stat':'percent',
    'bins':n_bins
}
# features= df_train.columns
cols=['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude',"MedHouseVal"]
fig, ax = plt.subplots(3,3, figsize=(16, 10))
ax = ax.flatten()

for i, column in enumerate(cols):
    sns.histplot(
        df_train[column], label='Train',
        ax=ax[i], color='red', **histplot_hyperparams
    )

**None Of Them Are Perfectly  Normally Distributed . Most Of Them Are Right Skewed .**

# <h1 style='background:#3b3a30; border:2; border-radius: 10px;padding-top: 2%;; font-size:200%; font-weight: bold; color:#c1502e'><center>Handling Skewdata</center></h1> 

**What is Skewed Data?**

> A data is called as skewed when curve appears distorted either to the left or to the right, in a statistical distribution.I.e one tail is longer than the other.

Such data can be handled by following ways:

-  Log Transform
-  Box Cox Transform
-  Square Root Transform



## **Log Transform**
-  This is most commonly used
-  Easily done by np.log()
-  Data should not have null values
-  Handle values at 0 (np.log(0) encounters divide by zero)

In [ ]:
# initial skewness on 'AveRooms'col
#as this is more correlated col 

t=sns.distplot(df_train["AveRooms"],label="Skewness: %.2f"%(df_train["AveRooms"].skew()) )
t.legend()

In [ ]:
# after log-transform

Log_Ave = df_train["AveRooms"].map(lambda i: np.log(i) if i > 0 else 0)
t=sns.distplot(Log_Ave,label="Skewness: %.2f"%(Log_Ave.skew()) )
t.legend()

Skewness is reduced from 1.30 to -0.33

## Box Cox Transform

-  Here Data must be positive

In [ ]:
# from scipy import stats
Boxcox= stats.boxcox(df_train["AveRooms"])
Boxcox= pd.Series(Boxcox[0])
t=sns.distplot(Boxcox,label="Skewness: %.2f"%(Boxcox.skew()) )
t.legend()

Here Skewness reduced from 1.30 to 0.03

## Square Root Transform

- Not generally used as taking square root shortens the range of variables
- Can be applied via np.sqrt()

In [ ]:
Sqrt_Ave = df_train["AveRooms"].map(lambda i: np.sqrt(i))
t=sns.distplot(Sqrt_Ave,label="Skewness: %.2f"%(Sqrt_Ave.skew()) )
t.legend()

Here Skewness reduced from 1.30 to 0.28

# <h1 style='background:#3b3a30; border:2; border-radius: 10px;padding-top: 2%;; font-size:200%; font-weight: bold; color:#c1502e'><center>Handling Outliers</center></h1> 

<img src="https://media.tenor.com/4T5hjh17a18AAAAM/sanditon-theojames.gif" width= 450 height = 400 />


**An outlier is an observation that is unlike the other observations. It is rare, or distinct, or does not fit in some way. It is also called anomalies.**

Outliers can have many causes, such as:
- Measurement or input error.
- Data corruption.
- True outlier observation.

There is no precise way to define and identify outliers in general because of the specifics of each dataset. Instead, you, or a domain expert, must interpret the raw observations and decide whether a value is an outlier or not.

- Nevertheless, we can use **statistical methods** to identify observations that appear to be rare or unlikely given the available data.
- This does not mean that **the values identified are outliers** and should be removed.
- A good tip is to consider plotting the identified outlier values, perhaps in the context of       non-outlier values to see if there are any **systematic relationships or patterns** to the           outliers. 

- If there is, perhaps they are not outliers and can be explained, or perhaps the outliers themselves can be identified more systematically.

#### **Types of outliers:**
Outlier can be of two types:
1) Univariate
2) Multivariate.
- Univariate outliers can be found when we look at distribution of a **single variable.**
- Multi-variate outliers are outliers in an **n-dimensional space**. In order to find them, you have to look at distributions in multi-dimensions.

#### **Impact Of Outliers:**
-  It increases the **error variance and reduces the power** of statistical tests
-  If the outliers are non-randomly distributed, they can **decrease normality**
-  They can bias or influence estimates that may be of substantive interest
-  They can also **impact the basic assumption** of Regression, ANOVA and other statistical model assumptions.

## **Outlier Detection Techniques:**
### **A) Univariate Outliers:**
# **1) Interquartile Range Method:**

- The concept of the Interquartile Range **(IQR) is used to build the boxplot graphs.** IQR is a concept in statistics that is used to measure the statistical dispersion and data variability by dividing the dataset into quartiles.

- In simple words, any dataset or any set of observations is divided into four defined intervals based upon the values of the data and how they compare to the entire dataset. **A quartile is what divides the data into three points and four intervals.**

- It is the difference between the third quartile and the first quartile (IQR = Q3 -Q1). Outliers in this case are defined as the observations that are below (Q1 − 1.5x IQR) or boxplot lower whisker or above (Q3 + 1.5x IQR) or boxplot upper whisker. It can be visually represented by the box plot.
<img src="https://www.thoughtco.com/thmb/S9Y2kXCgZdO4ics_6mWXUjNnmAg=/1500x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/boxplotwithoutliers-5b8ec88846e0fb0025192f90.jpg" width= 450 height = 400 />


In [ ]:
# func to Interquartile Range Method
def out_iqr(df , column):
    global lower,upper
    q25, q75 = np.quantile(df[column], 0.25), np.quantile(df[column], 0.75)
    # calculate the IQR
    iqr = q75 - q25
    # calculate the outlier cutoff
    cut_off = iqr * 1.5
    # calculate the lower and upper bound value
    lower, upper = q25 - cut_off, q75 + cut_off
    print('The IQR is',iqr)
    print('The lower bound value is', lower)
    print('The upper bound value is', upper)
    # Calculate the number of records below and above lower and above bound value respectively
    df1 = df[df[column] > upper]
    df2 = df[df[column] < lower]
    return print('Total number of outliers are', df1.shape[0]+ df2.shape[0])


In [ ]:
out_iqr(df_train,'AveRooms')
plt.figure(figsize = (10,6))
sns.distplot(df_train.AveRooms, kde=False)
plt.axvspan(xmin = lower,xmax= df_train.AveRooms.min(),alpha=0.2, color='red')
plt.axvspan(xmin = upper,xmax= df_train.AveRooms.max(),alpha=0.2, color='red')

Total number of outliers **are 565 .**
- Here the **red zone represents the outlier zone!** The records present in that zone are considered as outliers .
- **Remedial Measure:**
>Remove the records which are above the upper bound value and records below the lower bound value!

# **2. Z-Score method:**
- The Z-score is the **signed number of standard deviations** by which the value of an observation or data point is above the mean value of what is being observed or measured.
- The intuition behind Z-score is **to describe any data point by finding their relationship with     the Standard Deviation and Mean of the group of data points.** Z-score is finding the               distribution of data where mean is 0 and standard deviation is 1 i.e. normal distribution.

- while calculating the Z-score we **re-scale and center the data and look for data points** which are too far from zero. These data points which are way too **far from zero will be treated as the outliers**. In most of the cases a threshold of **3 or -3 is used** i.e if the Z-score value is greater than or less than 3 or -3 respectively, that data point will be identified as outliers.

- This technique assumes a **Gaussian distribution of the data.** The outliers are the data points that are in the tails of the distribution and therefore far from the mean. How far depends on a set threshold zthr for the normalized data points zi calculated with the formula:

**Z_score= (Xi - mean) / standard deviation**

where **Xi is a data point**, **'mean' is the mean of all X** and **'standard deviation' the standard deviation of all X.**

- An outlier is then a normalized data point which has an absolute value greater than Zthr. That is:
### **|Z_score| > Zthr**

- Commonly used Zthr values are 2.5, 3.0 and 3.5. Here we will be using 3.0
- For example, I'll take up the TMedical Cost Personal Datasets for explaining Z-Score method.

In [ ]:
# applying Z-Score on MedInc col

plt.figure(figsize = (10,5))
sns.distplot(df_train['MedInc'])

In [ ]:
def out_zscore(data):
    global outliers,zscore
    outliers = []
    zscore = []
    threshold = 3
    mean = np.mean(data)
    std = np.std(data)
    for i in data:
        z_score= (i - mean)/std 
        zscore.append(z_score)
        if np.abs(z_score) > threshold:
            outliers.append(i)
    return print("Total number of outliers are",len(outliers))

In [ ]:
out_zscore(df_train.MedInc)
plt.figure(figsize = (10,5))
sns.distplot(zscore)
plt.axvspan(xmin = 3 ,xmax= max(zscore),alpha=0.2, color='red')

### B) **MULTIVARIATE OUTLIERS:**

# 3. **DBSCAN (Density-Based Spatial Clustering of Applications with Noise):**

- This is a **clustering algorithm (an alternative to K-Means)** that clusters points together and identifies any points not belonging to a cluster as outliers. 

- It’s like K-means, except the number of clusters does not need to be specified in advance .
<img src="https://pic3.zhimg.com/v2-24f949e7d2fc310fba0bcd3d9441beb2_r.jpg" width= 450 height = 400 />


In [ ]:
# implementation of DBSCAN
X_db = df_train[['MedInc','AveRooms']].values

db = DBSCAN(eps=3.0, min_samples=10).fit(X_db)
labels = db.labels_
pd.Series(labels).value_counts()

**here -1 is representing outliers. We have 11 in this case .Let's visualize them**

In [ ]:
plt.figure(figsize=(12,12))

unique_labels = set(labels)
colors = ['blue', 'red']

for color,label in zip(colors, unique_labels):
    sample_mask = [True if l == label else False for l in labels]
    plt.plot(X_db[:,0][sample_mask], X_db[:, 1][sample_mask], 'o', color=color);
plt.xlabel('MedInc');
plt.ylabel('AveRooms');

# 4.**Local Outlier Factor Method(LOF):**
- **LOF uses density-based outlier detection** to identify local outliers, points that are outliers with respect to their local neighborhood, rather than with respect to the global data distribution. 
- **The higher the LOF value for an observation, the more anomalous the observation.**
- This is useful because **not all methods will not identify** a point that’s an outlier relative to a nearby cluster of points (a local outlier) if **that whole region is not an outlying region** in the global space of data points.
<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_lof_outlier_detection_001.png" width= 450 height = 400 />


In [ ]:
lof = LocalOutlierFactor(n_neighbors=50, contamination='auto')
lof_val= df_train[['AveBedrms','Population']].values
y_pred = lof.fit_predict(lof_val)
plt.figure(figsize=(12,12))
# plot the level sets of the decision function

in_mask = [True if l == 1 else False for l in y_pred]
out_mask = [True if l == -1 else False for l in y_pred]

plt.title("Local Outlier Factor (LOF)")
# inliers
a = plt.scatter(lof_val[in_mask, 0], lof_val[in_mask, 1], c = 'blue',
                edgecolor = 'k', s = 30)
# outliers
b = plt.scatter(lof_val[out_mask, 0], lof_val[out_mask, 1], c = 'red',
                edgecolor = 'k', s = 30)
plt.axis('tight')
plt.xlabel('AveBedrms');
plt.ylabel('Population');
plt.show()

In [ ]:
# another IRT method
n_bins = 50

# features= df_train.columns
cols=['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude',"MedHouseVal"]
fig, ax = plt.subplots(3,3, figsize=(16, 10))
ax = ax.flatten()

for i, column in enumerate(cols):
    sns.boxplot(
        df_train[column], 
        ax=ax[i], color='red'
    )

In [ ]:
# Let's take AveRooms for consideration
#Initial state with outlier

sns.boxplot(x='AveRooms',data=df_train)


In [ ]:
#removing some outliers

first_quartile=df_train.AveRooms.quantile(0.25)
third_quartile=df_train.AveRooms.quantile(0.75)
IQR = third_quartile - first_quartile


In [ ]:
new_boundary = third_quartile + 3 * IQR 

df_train.drop( df_train [ df_train ["AveRooms"] > new_boundary].index , axis=0 , inplace=True)

In [ ]:
# after outlier removal
sns.boxplot(x='AveRooms',data=df_train)

# <h1 style='background:#3b3a30; border:2; border-radius: 10px;padding-top: 2%;; font-size:200%; font-weight: bold; color:#c1502e'><center>Baseline Model</center></h1> 

- 1.Extreme Gradient Boosting (XGBoost)
- 2.LightGBM (Light Gradient Boosting Machine)

In [ ]:
X = df_train.drop('MedHouseVal', axis=1).copy()
y = df_train["MedHouseVal"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

preds = []
met_score = []
feature_importance_df = pd.DataFrame()

for fold, (idx_train, idx_valid) in enumerate(kf.split(X)):
    X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]
    X_valid, y_valid = X.iloc[idx_valid], y.iloc[idx_valid]
    
    model = XGBRegressor(booster='gbtree',
                         eval_metric='rmse',
                         early_stopping_rounds=100)
    
    model.fit(X_train, y_train,
              eval_set=[(X_valid, y_valid)],
              verbose=False)
    
    pred_valid = model.predict(X_valid)
    score = mean_squared_error(y_valid, pred_valid, squared=False)
    met_score.append(score)
    
  
    print(f"Fold: {fold + 1} Score: {score}")
    print('||'*20)
    
    test_preds = model.predict(df_test)
    preds.append(test_preds)

print(f"Overall Validation Score: {np.mean(met_score)}")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

preds2 = []
met_score2 = []


for fold, (idx_train, idx_valid) in enumerate(kf.split(X)):
    X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]
    X_valid, y_valid = X.iloc[idx_valid], y.iloc[idx_valid]
    
    model = LGBMRegressor(learning_rate=0.025, n_estimators=100_000, metric='rmse')
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgbm.early_stopping(500, verbose=True)])
    
    pred_valid = model.predict(X_valid)
    score = mean_squared_error(y_valid, pred_valid, squared=False)
    met_score2.append(score)
    
  
    print(f"Fold: {fold + 1} Score: {score}")
    print('||'*20)
    
    test_preds = model.predict(df_test)
    preds2.append(test_preds)

print(f"Overall Validation Score: {np.mean(met_score2)}")

In [ ]:
# Taking weighted avg of 2 preds

prediction1 = np.mean(np.column_stack(preds),axis=1)
prediction2 =np.mean(np.column_stack(preds2),axis=1)
final_pred=(prediction1+prediction2)/2

In [ ]:
final_pred

In [ ]:
df_sub['MedHouseVal'] = final_pred
# df_sub.to_csv('submission1.csv', index=False)
df_sub.head()

<img src="https://media1.giphy.com/media/nUSgpi4vHWqDC/giphy.gif" width= 550 height = 350 />

### **If you liked this kernel , Then Make an upvote👍. See You All With Another One .**
Some resources are collected from RADEK's & KASHIF's kernel .